In [ ]:
# !ffmpeg -decoders | grep -i nvidia

In [ ]:
# !pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu128
# !pip install torchcodec==0.9.1 --index-url=https://download.pytorch.org/whl/cu128

In [ ]:
# !pip install --upgrade pandas numpy tqdm

In [ ]:
# !wget -q https://github.com/AlexeyLunyakov/itmo-sc-tasks/raw/refs/heads/main/assignments/g1/model.py -O model.py
# !wget -q https://github.com/AlexeyLunyakov/itmo-sc-tasks/raw/refs/heads/main/assignments/g1/best_ctc.pt -O best_ctc.pt

In [ ]:
import sys
sys.path.append('.')

In [ ]:
import model as asr_model
import torch
import pandas as pd
from pathlib import Path

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, feature_extractor = asr_model.load_model_from_checkpoint("best_ctc.pt", device=device)

In [ ]:
# TEST_CSV = "/kaggle/input/asr-data/test.csv"
# DATA_ROOT = Path("/kaggle/input/competitions/asr-2026-spoken-numbers-recognition-challenge").resolve()
DATA_ROOT = Path("./assets/asr_2026").resolve()
TEST_CSV = DATA_ROOT / "test.csv"
test_df = pd.read_csv(TEST_CSV)

In [6]:
print("test_df columns:", test_df.columns.tolist())
print("test samples:", len(test_df))
display(test_df.head(5))

test_df columns: ['filename', 'ext', 'samplerate']
test samples: 2582


,filename,ext,samplerate
0,test/d2440788a9.mp3,mp3,16000
1,test/e247dbf761.mp3,mp3,16000
2,test/071f4a5be7.mp3,mp3,16000
3,test/798bd15db7.mp3,mp3,16000
4,test/58c0464ad5.mp3,mp3,16000


In [7]:
predictions = []
for idx, row in test_df.iterrows():
    audio_path = DATA_ROOT / row["filename"]
    pred = asr_model.predict_audio_file(model, feature_extractor, audio_path, device)
    predictions.append(pred)

/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(


In [8]:
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "transcription": predictions
})

In [9]:
# submission.to_csv("submission.csv", index=False)
print(submission.head())

              filename transcription
0  test/d2440788a9.mp3        461694
1  test/e247dbf761.mp3        207723
2  test/071f4a5be7.mp3         79187
3  test/798bd15db7.mp3         64048
4  test/58c0464ad5.mp3        861168
